# Mcardiac training

This public notebook was migrated from the audited read-only research source. 
Configure paths in `config.yaml` before execution. Time is metadata and is never a model input.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'Mcardiac'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


# End-to-end corrected four-fate lineage: E9.5h → E11.5h

Train the shared 3-D Stage 1 model, export its trajectory, build voxel boundaries, train the corrected-lineage Stage 2 model, and produce exactly 21 rollout frames. Time is metadata and is never passed as a model input.

The four Stage 2 fates remain migration/no event, differentiation through `AlphaTransitionNet`, birth, and death. The canonical scanVI AnnData object and the route-specific full-gene decoder are reused.


## 1. Reproducible setup

Configuration is centralized here. Every execution creates a fresh Stage2 directory, so a checkpoint from the broken implementation cannot be reused.

In [ ]:
from __future__ import annotations
import inspect
import json
import random
import time
from datetime import datetime

import numpy as np
import pandas as pd
import scanpy as sc
import torch

from stvirtual.models import stage1_3d as s1
from stvirtual.models import stage2_3d_lineage as s2
from stvirtual.utils.stage1_results import save_res as save_stage1_results

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
ROUTE = "E9.5h_to_E11.5h"
SRC, TGT = ROUTE.split("_to_")
ROUTE_IDS = [SRC, TGT]
STEPS, STAGE1_EPOCHS, STAGE2_EPOCHS = 20, 100, 100
assert STEPS + 1 == 21

DATA = SCANVI_ADATA
DIFF_CSV = DIFF_MAP
STAGE1_CKPT = STAGE1_CHECKPOINT_ROOT / ROUTE / "checkpoints" / "best.pt"
STAGE1_TRACE = CHECKPOINT_ROOT / "stage1_res" / f"rollout_stage1_{ROUTE}.npz"
DECODER_CKPT = experiment_path(CONFIG["decoder_checkpoint"].format(src=SRC, tgt=TGT))
RUN_DIR = RUN_ROOT / f"{ROUTE}_{datetime.now():%Y%m%d_%H%M%S}"
CKPT_DIR = STAGE2_CHECKPOINT_ROOT
FRAME_DIR = RUN_DIR / "rollout"
for directory in (RUN_DIR, FRAME_DIR):
    directory.mkdir(parents=True, exist_ok=False)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

required_inputs = [DATA, LR_PAIRS, DIFF_CSV, DECODER_CKPT]
missing = [str(path) for path in required_inputs if path is None or not path.exists()]
if missing:
    raise FileNotFoundError("Missing inputs:\n" + "\n".join(missing))
print("Fresh output:", RUN_DIR)


## 2. Train the shared 3-D Stage 1 model

Stage 1 reads the precomputed `X_scanVI` representation from the canonical AnnData object. It has no time input and uses the current `uot_*` loss interface.


In [ ]:
adata = sc.read_h5ad(DATA)
required_obs = {"stage", "mapped_celltype", "cx_aligned", "cy_aligned", "cz_aligned"}
missing_obs = sorted(required_obs - set(adata.obs.columns))
if missing_obs:
    raise KeyError(f"Canonical AnnData is missing obs columns: {missing_obs}")
if "X_scanVI" not in adata.obsm:
    raise KeyError("Canonical AnnData is missing adata.obsm['X_scanVI']")
if "counts" not in adata.layers:
    raise KeyError("Canonical AnnData is missing adata.layers['counts']")
missing_stages = sorted(set(ROUTE_IDS) - set(adata.obs["stage"].astype(str)))
if missing_stages:
    raise ValueError(f"Canonical AnnData is missing stages: {missing_stages}")

adata


In [ ]:
stage1_results = s1.train_model_multislice(
    model=None,
    adata_all=adata,
    slice_key="stage",
    route_ids=ROUTE_IDS,
    save_root=str(STAGE1_CHECKPOINT_ROOT),
    x_key="cx_aligned",
    y_key="cy_aligned",
    z_key="cz_aligned",
    latent_key="X_scanVI",
    steps=STEPS,
    guide_eps=0.01,
    uot_eps=0.05,
    uot_tau=0.5,
    uot_lam_x=2.0,
    uot_lam_f=0.05,
    guide_topk=512,
    guide_temp=0.5,
    guide_schedule="linear",
    terrain_gap=False,
    cell_type_key="mapped_celltype",
    lam_context=10.0,
    lam_residual=1.0,
    lam_vsmooth=0.1,
    lam_uot=1.0,
    lib_layer="counts",
    latent_dim=int(adata.obsm["X_scanVI"].shape[1]),
    epochs=STAGE1_EPOCHS,
    device=str(device),
)
assert ROUTE in stage1_results
assert STAGE1_CKPT.is_file(), STAGE1_CKPT


In [ ]:
saved_stage1 = save_stage1_results(
    res1=stage1_results,
    adata_all=adata,
    out_dir=str(STAGE1_TRACE.parent),
    slice_key="stage",
    ann_key="mapped_celltype",
    save_prefix="rollout_stage1",
    steps=STEPS,
    n_cache=256,
    unnormalize=False,
)
STAGE1_TRACE = Path(saved_stage1[ROUTE]).resolve()
assert STAGE1_TRACE.is_file()
print("Stage-1 checkpoint:", STAGE1_CKPT)
print("Stage-1 trace:", STAGE1_TRACE)


## 3. Generate the 3-D voxel boundaries

In [ ]:
from stvirtual.utils import boundary_3d as boundary

boundary_cfg = CONFIG["boundary"]
trace_data = np.load(STAGE1_TRACE, allow_pickle=True)
frames = [np.asarray(frame, dtype=np.float32) for frame in trace_data["coords_frames"]]
BOUND_DIR = RUN_ROOT / "bound_3d" / ROUTE
BOUND_DIR.mkdir(parents=True, exist_ok=True)
bbox = boundary.compute_bbox_3d_from_frames(frames, margin=boundary_cfg["margin"])
D, H, W, grid_info = s2.auto_choose_grid_size_3d(
    np.concatenate(frames), pts_per_cell=boundary_cfg["pts_per_cell"], base=boundary_cfg["base"],
    max_hw=boundary_cfg["max_hw"], min_hw=boundary_cfg["min_hw"],
    max_d=boundary_cfg["max_d"], min_d=boundary_cfg["min_d"], margin_xyz=boundary_cfg["margin"],
)
for frame_index, coordinates in enumerate(frames):
    left, right = max(0, frame_index - 1), min(len(frames), frame_index + 2)
    neighborhood = np.concatenate(frames[left:right])
    volume = boundary.voxelize_points_3d_fixed_bbox_dense(
        neighborhood, bbox=bbox, D=D, H=H, W=W,
        splat_radius=boundary_cfg["splat_radius"], dilate_iter=boundary_cfg["dilate_iter"],
        close_iter=boundary_cfg["close_iter"], fill_holes=boundary_cfg["fill_holes"],
        keep_lcc=boundary_cfg["keep_lcc"], min_count=boundary_cfg["min_count"],
    )
    boundary.save_voxel_volume(volume, BOUND_DIR / f"bound_z{frame_index:03d}.npz")
    boundary.save_voxel_preview_png(volume, BOUND_DIR / f"bound_z{frame_index:03d}.png")
print("3D boundary grid:", (D, H, W), grid_info)


## 4. Build the corrected-lineage context and verify diffmap

Checks prove Stage2 receives a non-empty diffmap and that self-edges are removed.

In [ ]:
ctx = s2.build_global_ctx(adata_path=str(DATA), lr_pairs_path=str(LR_PAIRS), ckpt_3dslice=str(STAGE1_CKPT), device=device, layer_col="mapped_celltype")
stage = s2.StageCfg(src=SRC, tgt=TGT, out_npz_path=str(STAGE1_TRACE), bound_dir=str(BOUND_DIR), decoder_checkpoint=str(DECODER_CKPT), scanvi_dir=None, model_type="scanvi", use_lr=True, diff_csv=str(DIFF_CSV), lr_source="decoder", use_latent=True, latent_key="X_scanVI", z_csv_offset=0, layer_col="mapped_celltype")
pack_audit = s2.prepare_one_stage(ctx, stage, sample_key="stage", layer_col="mapped_celltype")
assert pack_audit.T == STEPS
assert pack_audit.diff_tgt_idx is not None and pack_audit.diff_tgt_w is not None
for src_idx in range(pack_audit.n_layers):
    assert not bool((pack_audit.diff_tgt_idx[src_idx] == src_idx).any())
print("layers/diff shape:", pack_audit.n_layers, tuple(pack_audit.diff_tgt_idx.shape))


In [ ]:
source = inspect.getsource(s2)
reward_contract = [
    "occ_loss = occ_l1.view(1) + float(rl_occ_iou) * iou_loss",
    "final_loss = tgt_loss + float(rl_occ) * occ_loss",
    "reward = -final_loss",
]
assert all(line in source for line in reward_contract)
reward_block = source[source.index(reward_contract[0]):source.index("r_det = reward.detach()")]
assert "composition" not in reward_block
print("Reward contract unchanged")


## 5. Train a fresh continuous-alpha Stage 2

Birth/death/migration and the original reward are retained. AlphaNet samples continuous source→middle and middle→target rates; their likelihoods enter REINFORCE, while commit occurs deterministically at continuous progress alpha ≥ 0.80.

In [ ]:
training_started = time.perf_counter()
outs = s2.run_multi_stages(ctx=ctx, rl_xy=2.0, rl_z=0.05, sample_key="stage", stages=[stage], best_ckpt_dir=str(CKPT_DIR), train_kwargs=dict(EPOCHS=STAGE2_EPOCHS, LR=1e-4, TAU_DIFF=1.0))
training_seconds = time.perf_counter() - training_started
assert len(outs) == 1
train_out = outs[0]["train_out"]
best_ckpt = Path(outs[0]["best_ckpt_path"])
assert best_ckpt.exists()
history = pd.DataFrame(train_out["history"])
history.to_csv(RUN_DIR / "training_history.csv", index=False)
checkpoint = torch.load(best_ckpt, map_location="cpu")
assert checkpoint["corrected_implementation_id"] == s2.CORRECTED_IMPLEMENTATION_ID
assert checkpoint["commit_completion_threshold"] == 0.80
assert checkpoint["alpha_min_reference_rate"] == 0.20
assert checkpoint["alpha_reference_steps"] == 100
if s2.alpha_gradient_required():
    assert checkpoint["alpha_grad_norm"] > 0 and checkpoint["alpha_param_delta"] > 0
else:
    assert checkpoint["alpha_grad_norm"] == 0 and checkpoint["alpha_param_delta"] == 0
print(history.tail())
print("alpha grad/delta:", checkpoint["alpha_grad_norm"], checkpoint["alpha_param_delta"])


## 6. Fixed-seed 21-frame rollout

Frame 0 is the source; frame 20 is the formal endpoint.

In [ ]:
rollout_started = time.perf_counter()
rollout = s2.rollout_policy_one_stage(s2, ctx, stage, str(best_ckpt), sample_key="stage", seed=SEED, ADVECT_LATENT=True, TAU_BIRTH=1.0, TAU_DEATH=1.0, TAU_DIFF=1.0, output_dir=FRAME_DIR, output_prefix=ROUTE)
rollout_seconds = time.perf_counter() - rollout_started
assert rollout["T"] == STEPS
assert len(rollout["coords"]) == len(rollout["layers"]) == 21
np.savez_compressed(RUN_DIR / "rollout_21_frames.npz", **{k: np.asarray(v, dtype=object) for k, v in rollout.items() if isinstance(v, list)}, t=np.asarray(rollout["t"], dtype=np.float32), T=np.asarray([rollout["T"]], dtype=np.int32))
print("frames:", len(rollout["coords"]))


## Persist lineage metadata in each H5AD frame

Write `uid`, `parent_uid`, `diff_alpha`, and UID-tracked source/target layer IDs and names into `adata.obs`. Existing notebook outputs are preserved.


In [ ]:
LINEAGE_OBS_FIELDS = (
    "uid",
    "parent_uid",
    "diff_alpha",
    "src_layer_id",
    "src_layer",
    "tgt_layer_id",
    "tgt_layer",
)

def persist_lineage_rollout_obs(rollout_result, celltype_names):
    output_paths = rollout_result.get("output_paths")
    required = ("uid", "parent_uid", "layers", "is_diff", "diff_alpha", "diff_tgt_layer", "commit_step")
    missing = [field for field in required if not isinstance(rollout_result.get(field), list)]
    if missing:
        raise KeyError(f"rollout_result is missing lineage fields: {missing}")
    if not isinstance(output_paths, list):
        raise KeyError("rollout_result has no output_paths; pass output_dir to rollout_policy_one_stage")
    frame_count = len(rollout_result["coords"])
    if len(output_paths) != frame_count:
        raise ValueError("output_paths and rollout frames have different lengths")

    names = [str(name) for name in celltype_names]
    source_by_uid = {}
    target_by_uid = {}
    for frame_index in range(frame_count):
        uids = np.asarray(rollout_result["uid"][frame_index], dtype=np.int64)
        current_layers = np.asarray(rollout_result["layers"][frame_index], dtype=np.int64)
        target_layers = np.asarray(rollout_result["diff_tgt_layer"][frame_index], dtype=np.int64)
        for uid, current_layer, target_layer in zip(uids, current_layers, target_layers):
            source_by_uid.setdefault(int(uid), int(current_layer))
            if int(target_layer) >= 0:
                target_by_uid[int(uid)] = int(target_layer)

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        uids = np.asarray(rollout_result["uid"][frame_index], dtype=np.int64)
        parent_uids = np.asarray(rollout_result["parent_uid"][frame_index], dtype=np.int64)
        diff_alpha = np.asarray(rollout_result["diff_alpha"][frame_index], dtype=np.float32)
        is_diff = np.asarray(rollout_result["is_diff"][frame_index], dtype=bool)
        commit_step = np.asarray(rollout_result["commit_step"][frame_index], dtype=np.int32)
        has_lineage = is_diff | (commit_step >= 0)

        src_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        tgt_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        for index, uid in enumerate(uids):
            if has_lineage[index]:
                src_ids[index] = source_by_uid[int(uid)]
                tgt_ids[index] = target_by_uid.get(int(uid), -1)

        src_names = np.array([names[index] if 0 <= index < len(names) else "" for index in src_ids], dtype=str)
        tgt_names = np.array([names[index] if 0 <= index < len(names) else "" for index in tgt_ids], dtype=str)
        frame_adata.obs["uid"] = uids
        frame_adata.obs_names = uids.astype(str)
        frame_adata.obs["parent_uid"] = parent_uids
        frame_adata.obs["diff_alpha"] = diff_alpha
        frame_adata.obs["src_layer_id"] = src_ids
        frame_adata.obs["src_layer"] = src_names
        frame_adata.obs["tgt_layer_id"] = tgt_ids
        frame_adata.obs["tgt_layer"] = tgt_names
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": frame_count, "obs_fields": list(LINEAGE_OBS_FIELDS)}

rollout_obs_report = persist_lineage_rollout_obs(rollout, ctx.layers_list)
rollout_obs_report


## Outputs

Canonical rollout frames and `celltype_mapping.csv` are written by the public rollout adapter.
